# 📚 Resum de Funcions – Pràctica 4: Descriptors d'Imatge, Punts Clau & Textura (i fonaments P2-P3)

Aquest notebook recull totes les funcions, eines i algorismes de biblioteques externes utilitzats a les pràctiques de Visió per Computador (actualitzat per a la **Pràctica 4**, consolidant i ampliant tot el contingut de la **Pràctica 2** i la **Pràctica 3**).

Inclou explicacions teòriques resumides, paràmetres clau, valors recomanats, tipus de dades i exemples d'ús pràctics i reproduïbles.

**Biblioteques principals:**
- `scikit-image` (`skimage`): mòduls `io`, `color`, `transform`, `exposure`, `filters`, `morphology`, `util`, `feature` (HOG, CENSURE, ORB, SIFT, match_descriptors, plot_matched_features), `segmentation`
- `scikit-learn` (`sklearn`): mòduls `cluster` (`AgglomerativeClustering`), `feature_extraction.image` (`grid_to_graph`)
- `scipy`: mòdul `ndimage` (`convolve`), `signal` (`convolve2d`)
- `util.lm`: mòdul per generar el banc de filtres de Leung–Malik (`makeLMfilters`)
- `numpy` (`np`): àlgebra matricial, mètriques L2, ordenació (`argsort`), padding, mediana temporal
- `matplotlib.pyplot` (`plt`) & `matplotlib.patches`: visualització avançada, deteccions amb capses (`Rectangle`) i comparatives d'emparellament


---
## 0. Imports generals i funcions auxiliars essencials (`ensure_gray`, `normalize`)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import os
import glob
from pathlib import Path

# Configuració per a visualitzacions clares
plt.rcParams['figure.dpi'] = 100
%matplotlib inline

# ══════════════════════════════════════════════════════════════════
# Funcions auxiliars recomanades a la P4 per a robustesa de codi:
# ══════════════════════════════════════════════════════════════════

def normalize(img):
    """
    Normalitza els valors d'intensitat de la imatge a l'interval [0, 1] (Min-Max).
    Evita divisions per zero si la imatge té valor constant.
    """
    img = img.astype(float)
    min_val = img.min()
    max_val = img.max()
    if max_val == min_val:
        return np.zeros_like(img)
    return (img - min_val) / (max_val - min_val)

def ensure_gray(img):
    """
    Garanteix que la imatge estigui en escala de grisos 2D (H, W).
    - Si és 2D: la retorna sense canvis.
    - Si té canal alfa (RGBA, shape (H, W, 4)): elimina l'alfa i converteix RGB a grisos.
    - Si és RGB (shape (H, W, 3)): converteix a grisos amb skimage.color.rgb2gray.
    """
    from skimage.color import rgb2gray
    if img.ndim == 2:
        return img
    if img.ndim == 3 and img.shape[2] == 4:
        img = img[:, :, :3]
    if img.ndim == 3 and img.shape[2] == 3:
        img = rgb2gray(img)
    return img


---
## 1. `skimage.io` – Lectura i càrrega d'imatges


In [ ]:
from skimage import io

# io.imread llegeix la imatge i retorna un array de NumPy:
# - Imatge RGB → shape (H, W, 3), dtype uint8, rang [0, 255]
# - Imatge RGBA → shape (H, W, 4), dtype uint8, rang [0, 255]
# - Imatge grisos → shape (H, W), dtype uint8, rang [0, 255]
# - as_gray=True → converteix automàticament a escala de grisos float64 [0.0, 1.0]

try:
    img_starbucks = io.imread("images_p4/starbucks.jpg")
    img_dog = io.imread("images_p4/dog.jpg")
    print("Starbucks shape:", img_starbucks.shape, "| dtype:", img_starbucks.dtype)
    print("Dog shape:", img_dog.shape, "| dtype:", img_dog.dtype)
except FileNotFoundError:
    print("Comprova que executis el notebook des del directori que conté images_p4/")


---
## 2. `skimage.color` – Conversions de color


In [ ]:
from skimage.color import rgb2gray

# Converteix imatge RGB a escala de grisos mitjançant luminància ponderada:
# Y = 0.2125 R + 0.7154 G + 0.0721 B
# - Entrada: uint8 [0, 255] o float [0, 1] amb shape (H, W, 3)
# - Sortida: float64 [0.0, 1.0] amb shape (H, W)
# ⚠️ Atenció: Si la imatge és RGBA (4 canals), rgb2gray llança ValueError.
#    Feu servir sempre la funció `ensure_gray()` definida a la secció 0.

if 'img_dog' in locals():
    dog_gray = ensure_gray(img_dog)
    print("Imatge convertida amb ensure_gray → shape:", dog_gray.shape, "| dtype:", dog_gray.dtype)


---
## 3. `skimage` & `numpy` – Conversions de tipus (`img_as_float`, `img_as_ubyte`, `astype`)


In [ ]:
from skimage import img_as_float, img_as_ubyte

# img_as_float: converteix a float64 normalitzat a [0.0, 1.0] (uint8 es divideix entre 255)
# img_as_ubyte: converteix a uint8 [0, 255] (avís de clipping si valors fora de [0, 1])
# astype: canvia el tipus intern sense canviar els valors (no normalitza)

if 'img_dog' in locals():
    float_demo = img_as_float(img_dog)
    ubyte_demo = img_as_ubyte(dog_gray)
    print("img_as_float rang:", float_demo.min(), "-", float_demo.max(), "| dtype:", float_demo.dtype)
    print("img_as_ubyte rang:", ubyte_demo.min(), "-", ubyte_demo.max(), "| dtype:", ubyte_demo.dtype)


---
## 4. `skimage.transform` – Escala, redimensionat i rotació (`rescale`, `resize`, `rotate`)


In [ ]:
from skimage.transform import rescale, resize, rotate

# 1. rescale: escala proporcionalment per un factor
# 2. resize: força una mida exacta (H, W) (retorna sempre float64 [0.0, 1.0])
# 3. rotate: gira la imatge un cert angle en graus (sentit antihorari)
#    - resize=True: amplia el llenç per no retallar cantonades girades

if 'dog_gray' in locals():
    dog_240 = resize(dog_gray, (240, 240), anti_aliasing=True)
    dog_rot = rotate(dog_240, angle=30, resize=True)
    print("resize (240, 240) shape:", dog_240.shape)
    print("rotate 30° shape:", dog_rot.shape)


---
## 5. `skimage.exposure` – Histogrames i reescalat d'intensitat (`rescale_intensity`)


In [ ]:
from skimage.exposure import histogram, rescale_intensity

# exposure.histogram: retorna (hist, bin_centers)
# exposure.rescale_intensity: reescala el contrast d'una imatge a un cert rang.
# 💡 Molt utilitzat a la P4 per visualitzar la imatge de gradients HOG:
#    hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

if 'dog_gray' in locals():
    hist, bin_centers = histogram(dog_gray, nbins=32)
    print("Histogram nbins:", len(hist), "| bins rang:", bin_centers.min(), "-", bin_centers.max())


---
## 6. `scipy.ndimage`, `scipy.signal` & `skimage.filters` – Convolució i filtres espacials


In [ ]:
from scipy.ndimage import convolve
from scipy.signal import convolve2d
from skimage.filters import gaussian, median
from skimage.morphology import disk

# 1. scipy.ndimage.convolve: convolució N-dimensional (mode='reflect', 'constant', etc.)
# 2. scipy.signal.convolve2d: convolució 2D tradicional (mode='same', 'valid', boundary='symm')
# 3. gaussian: filtre de suavitzat continu (sigma)
# 4. median: filtre no lineal excel·lent per a soroll impulsiu amb footprint (disk, rectangle)

if 'dog_gray' in locals():
    dog_blur = gaussian(dog_gray, sigma=1.5)
    print("Filtre gaussià aplicat correctament")


---
## 7. `skimage.util.random_noise` – Addició de soroll sintètic


In [ ]:
from skimage.util import random_noise

# random_noise genera soroll sintètic i retorna SEMPRE float64 [0.0, 1.0]
# - mode='gaussian': soroll gaussià amb paràmetres 'mean' i 'var'
# - mode='s&p': soroll salt-and-pepper (impulsiu) amb paràmetre 'amount'
if 'dog_gray' in locals():
    dog_noisy = random_noise(dog_gray, mode='gaussian', var=0.01)
    print("Soroll gaussià afegit (dtype:", dog_noisy.dtype, ")")


---
## 8. `skimage.filters` & `skimage.feature` – Detecció de contorns (Sobel, Prewitt, Canny)


In [ ]:
from skimage.filters import sobel, sobel_h, sobel_v, prewitt
from skimage.feature import canny

# - sobel / prewitt: magnitud del gradient espacial
# - sobel_h / sobel_v: gradients parcials horitzontal (X) i vertical (Y)
# - canny: detector òptim amb hysterisi (retorna màscara booleana)
if 'dog_gray' in locals():
    edges = canny(dog_gray, sigma=1.2, low_threshold=0.1, high_threshold=0.2)
    print("Contorns detectats amb Canny:", np.sum(edges), "píxels")


---
## 9. `skimage.segmentation` – Segmentació d'imatges (Felzenszwalb, SLIC, Quickshift, `mark_boundaries`)


In [ ]:
from skimage.segmentation import felzenszwalb, slic, quickshift, mark_boundaries

# 1. felzenszwalb: segmentació basada en grafs (paràmetres: scale, sigma, min_size)
# 2. slic: K-Means en espai 5D (paràmetres: n_segments, compactness, sigma, start_label=1)
# 3. quickshift: cerca de modes local (paràmetres: kernel_size, max_dist, ratio)
# 4. mark_boundaries: superposa les vores dels segments sobre la imatge (retorna RGB float64)

if 'img_dog' in locals():
    seg_slic = slic(img_dog, n_segments=50, compactness=10, sigma=1.0, start_label=1)
    print("SLIC superpíxels calculats:", len(np.unique(seg_slic)))


---
## 10. `sklearn.cluster` & `grid_to_graph` – Clustering jeràrquic basat en grafs


In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.feature_extraction.image import grid_to_graph

# grid_to_graph(h, w): matriu d'adjacència de connectivitat 4-veïns
# AgglomerativeClustering: agrupament jeràrquic espacialment restringit (linkage='ward')
if 'dog_gray' in locals():
    small = resize(dog_gray, (30, 30), anti_aliasing=True)
    h_s, w_s = small.shape
    connectivity = grid_to_graph(h_s, w_s)
    clustering = AgglomerativeClustering(n_clusters=4, linkage='ward', connectivity=connectivity)
    labels = clustering.fit_predict(small.reshape(-1, 1)).reshape(h_s, w_s)
    print("Agglomerative clustering completat en mida reduïda:", labels.shape)


---
## 11. Subtracció de fons per a detecció de moviment (Background Subtraction)


In [ ]:
# Model de fons estàtic calculat mitjançant la mediana temporal:
# bck_image = np.median(training_frames, axis=0).astype(np.float64)
# img_diff = np.abs(test_frame - bck_image)
# binary_mask = img_diff > threshold

print("Subtracció de fons: es basa en mediana temporal (np.median) i llindarització de diferències")


---
## 12. Template Matching bàsic (SSD & NCC)


In [ ]:
from skimage.feature import match_template

# - SSD (Sum of Squared Differences): càlcul manual amb norma L2 ||sub_image - template||_2
# - NCC (Normalized Cross-Correlation): skimage.feature.match_template (màxims indiquen coincidència)
# - Padding per alinear mapes: np.pad(res, ((0, h_tpl - 1), (0, w_tpl - 1)), mode='edge')
# - Localització de pics: np.argwhere(mascara)

print("Template matching: SSD (mínims d'error) i NCC (màxims de correlació)")


---
## 13. `skimage.feature.hog` – Descriptor HOG i Detecció de Persones [NOU P4]

El descriptor **HOG (Histogram of Oriented Gradients)** és una eina clàssica fonamental per a la detecció d'objectes i persones basada en la forma i l'aparença local:

1. **Càlcul de gradients**: es calculen les derivades horitzontal i vertical de la imatge per obtenir la magnitud i orientació de cada píxel (0°-180° unsigned).
2. **Votació per cel·les (*Cells*)**: la imatge es divideix en petites cel·les espacials (ex: 8x8 o 4x4 píxels). Cada píxel vota a l'histograma d'orientacions de la seva cel·la proporcionalment a la magnitud del seu gradient.
3. **Normalització per blocs (*Blocks*)**: s'agrupen cel·les veïnes en blocs (ex: 2x2 o 3x3 cel·les) i es normalitzen els vectors resultants (normalització `L2-Hys` per defecte) per fer el descriptor invariant a canvis locals d'il·luminació i contrast.
4. **Detecció amb finestra lliscant (*Sliding Window*)**: per detectar un objecte (ex: vianant), es fa lliscar una finestra de detecció amb la mida del patró (*template*), s'extreu el descriptor HOG de cada regió i es calcula la distància euclidiana respecte al descriptor de referència. La regió amb la **distància mínima** correspon a la millor detecció.


In [ ]:
from skimage.feature import hog
from skimage import exposure

# ══════════════════════════════════════════════════════════════════
# 13.1 Càlcul i visualització del descriptor HOG
# ══════════════════════════════════════════════════════════════════
# Paràmetres clau de skimage.feature.hog:
# - orientations: nombre de bins d'orientació (típicament 8 o 9 bins).
# - pixels_per_cell: mida espacial de cada cel·la en píxels (ex: (8, 8) o (4, 4)).
# - cells_per_block: nombre de cel·les per bloc de normalització (ex: (2, 2) o (3, 3)).
# - block_norm: mètode de normalització de blocs ('L2-Hys' per defecte, 'L1', 'L2').
# - visualize=True: retorna la tupla (fd, hog_image) per poder visualitzar el camp vectorial.
# - feature_vector=True: aplana la sortida en un vector 1D unidimensional.

try:
    tpl_person = ensure_gray(io.imread("images_p4/person_template.bmp"))
    tpl_norm = normalize(tpl_person)

    # 1. HOG amb paràmetres per defecte (orientations=9, pixels_per_cell=(8,8), cells_per_block=(3,3))
    fd_def, hog_img_def = hog(tpl_norm, orientations=9, pixels_per_cell=(8, 8), cells_per_block=(3, 3), visualize=True)
    hog_img_def_rescaled = exposure.rescale_intensity(hog_img_def, in_range=(0, 10))

    # 2. HOG amb paràmetres personalitzats (orientations=8, pixels_per_cell=(4,4), cells_per_block=(2,2))
    fd_custom, hog_img_custom = hog(tpl_norm, orientations=8, pixels_per_cell=(4, 4), cells_per_block=(2, 2), visualize=True)
    hog_img_custom_rescaled = exposure.rescale_intensity(hog_img_custom, in_range=(0, 10))

    print(f"HOG per defecte: vector mida {fd_def.shape}")
    print(f"HOG personalitzat: vector mida {fd_custom.shape}")

    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(tpl_norm, cmap='gray')
    ax[0].set_title("Patró original")
    ax[0].axis('off')

    ax[1].imshow(hog_img_def_rescaled, cmap='gray')
    ax[1].set_title(f"HOG defecte (len={len(fd_def)})")
    ax[1].axis('off')

    ax[2].imshow(hog_img_custom_rescaled, cmap='gray')
    ax[2].set_title(f"HOG personalitzat (len={len(fd_custom)})")
    ax[2].axis('off')
    plt.tight_layout()
    plt.show()
except FileNotFoundError:
    print("images_p4/person_template.bmp no trobat")


In [ ]:
# ══════════════════════════════════════════════════════════════════
# 13.2 Detecció de persones amb Finestra Lliscant (Sliding Window)
# ══════════════════════════════════════════════════════════════════

def detect_person_hog(image, template, hog_params, step=8):
    """
    Detecta la millor posició del patró en una imatge usant finestra lliscant i HOG.
    Retorna: (x_best, y_best, min_dist, w, h)
    """
    h_tpl, w_tpl = template.shape
    h_img, w_img = image.shape

    if h_img < h_tpl or w_img < w_tpl:
        return None

    # Descriptor HOG del patró de referència
    fd_template = hog(template, **hog_params)

    best_x, best_y = 0, 0
    min_dist = np.inf

    # Finestra lliscant amb pas 'step'
    for y in range(0, h_img - h_tpl + 1, step):
        for x in range(0, w_img - w_tpl + 1, step):
            window = image[y : y + h_tpl, x : x + w_tpl]
            fd_win = hog(window, **hog_params)
            
            # Distància euclidiana entre descriptors HOG
            dist = np.linalg.norm(fd_template - fd_win)
            if dist < min_dist:
                min_dist = dist
                best_x, best_y = x, y

    return best_x, best_y, min_dist, w_tpl, h_tpl

# Prova ràpida si existeix la carpeta de test:
test_paths = glob.glob("images_p4/TestPersonImages/*.bmp")
if test_paths and 'tpl_norm' in locals():
    sample_img = normalize(ensure_gray(io.imread(test_paths[0])))
    params = {'orientations': 8, 'pixels_per_cell': (8, 8), 'cells_per_block': (2, 2)}
    res = detect_person_hog(sample_img, tpl_norm, params, step=16)
    if res:
        bx, by, d, wt, ht = res
        print(f"Detecció reeixida a x={bx}, y={by} amb distància HOG={d:.4f}")


---
## 14. Detectors de Punts Clau (Keypoints): `CENSURE` & `DoG` [NOU P4]

La detecció de punts clau (*keypoint detection*) cerca localitzacions destacades, repetibles i ben condicionades (cantonades, taques, centres d'estructures):

### 14.1 `skimage.feature.CENSURE` (STAR Detector)
- **CenSurE (CENsus TRansform hUncanny REgions)** és un detector multiescala altament eficient que aproxima el Laplacià de la Gaussiana (LoG) mitjançant filtres centre-perifèria (*center-surround bi-laplacian*).
- **Paràmetres clau**:
  - `min_scale` / `max_scale`: rang d'escales discretes a avaluar (escales de 1 a 7). Escales baixes detecten cantonades fines; escales altes detecten estructures grans.
  - `mode`: forma geomètrica dels filtres:
    - `'DoB'` (*Difference of Boxes*): quadrats simples, el més ràpid però menys invariant a rotació.
    - `'Octagon'`: octògons per aproximar cercles.
    - `'STAR'`: dues caixes creuades a 45°, excel·lent equilibri entre velocitat i invariància rotacional.
  - `non_max_threshold`: llindar de supressió de no-màxims (elimina respostes febles de baix contrast).
  - `line_threshold`: supressió de respostes lineals / vores (mètric similar a Harris per quedar-se només amb cantonades i taques reals).
- ⚠️ **Important**: CENSURE és **únicament un detector** de punts clau; **no** genera descriptors ni permet fer emparellament per si sol.


In [ ]:
from skimage.feature import CENSURE

# Aplicació de CENSURE sobre una imatge de prova
try:
    starbucks_gray = ensure_gray(io.imread("images_p4/starbucks4.jpg"))

    # Creació i execució del detector CENSURE
    censure_detector = CENSURE(
        min_scale=1, 
        max_scale=7, 
        mode='STAR', 
        non_max_threshold=0.15, 
        line_threshold=10
    )
    censure_detector.detect(starbucks_gray)

    keypoints_censure = censure_detector.keypoints
    scales_censure = censure_detector.scales

    print(f"Punts clau detectats per CENSURE: {len(keypoints_censure)}")
    print(f"Shape keypoints: {keypoints_censure.shape} (coordenades fila, columna)")

    plt.figure(figsize=(6, 6))
    plt.imshow(starbucks_gray, cmap='gray')
    plt.scatter(keypoints_censure[:, 1], keypoints_censure[:, 0], 2 * scales_censure ** 2, facecolors='none', edgecolors='r', linewidths=1.2)
    plt.title(f"CENSURE Keypoints (total: {len(keypoints_censure)})")
    plt.axis('off')
    plt.show()
except FileNotFoundError:
    print("images_p4/starbucks4.jpg no trobat")


---
## 15. Descriptors Locals i Emparellament: `ORB` & `SIFT` [NOU P4]

Per reconèixer i emparellar objectes entre imatges preses des de diferents perspectives, escales o rotacions, calen descriptors locals invariats:

### 15.1 `skimage.feature.ORB` (Oriented FAST and Rotated BRIEF)
- **Detector**: *oFAST* (troba cantonades ràpidament i calcula l'orientació mitjançant el centreide d'intensitat del pegat).
- **Descriptor**: *rBRIEF* (compara parelles d'intensitats de píxels girades segons l'angle calculat, generant una cadena de **256 bits / booleans**).
- **Distància**: **Hamming** (comptar bits diferents mitjançant operació XOR, extremadament ràpid).
- **Paràmetres**: `n_keypoints` (màxim de punts), `fast_threshold` (llindar FAST; com més baix, més punts).

### 15.2 `skimage.feature.SIFT` (Scale-Invariant Feature Transform)
- **Detector**: cerca d'extrems en l'espai d'escala continu creat per diferències de Gaussianes (*DoG*).
- **Descriptor**: histograma 3D de gradients orientats (4x4 cel·les amb 8 orientacions = **vector de 128 dimensions float**), alineat amb l'orientació dominant i normalitzat en L2 amb retall de contrast.
- **Distància**: **Euclidiana (L2)**.
- **Paràmetres**: `n_octaves` (nombre d'octaves de la piràmide), `n_scales` (escales per octava), `sigma_min` (sigma inicial).

### 15.3 `skimage.feature.match_descriptors`
- Troba correspondències entre dos conjunts de descriptors.
- `metric`: `'hamming'` per a descriptors binaris (ORB), `'euclidean'` per a descriptors continus (SIFT).
- `max_ratio`: **Lowe's ratio test** (quocient entre la distància al veí més proper i la del segon veí més proper; típic 0.75 - 0.8). Rebutja emparellaments ambigus.
- `cross_check=True`: exigeix coherència bidireccional (el punt A tria B com a millor parella i B tria A).
- Retorna una matriu d'índexs `(M, 2)` on `matches[:, 0]` apunta a `desc1` i `matches[:, 1]` apunta a `desc2`.

### 15.4 `skimage.feature.plot_matched_features`
- Visualitza les dues imatges de costat unint els punts emparellats amb línies de colors.


In [ ]:
from skimage.feature import ORB, SIFT, match_descriptors, plot_matched_features
import time

# ══════════════════════════════════════════════════════════════════
# Funcions completes d'extracció i emparellament per a ORB i SIFT
# ══════════════════════════════════════════════════════════════════

def match_orb_images(path1, path2, n_keypoints=800, fast_threshold=0.08, max_ratio=0.8, cross_check=True, plot=True):
    img1 = ensure_gray(io.imread(path1))
    img2 = ensure_gray(io.imread(path2))

    t0 = time.time()
    orb = ORB(n_keypoints=n_keypoints, fast_threshold=fast_threshold)
    
    orb.detect_and_extract(img1)
    kp1, desc1 = orb.keypoints, orb.descriptors

    orb.detect_and_extract(img2)
    kp2, desc2 = orb.keypoints, orb.descriptors

    # Emparellament amb distància Hamming per a descriptors binaris
    matches = match_descriptors(desc1, desc2, metric='hamming', max_ratio=max_ratio, cross_check=cross_check)
    elapsed = time.time() - t0

    if plot:
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_matched_features(img1, img2, keypoints1=kp1, keypoints2=kp2, matches=matches, ax=ax)
        ax.set_title(f"ORB: {len(matches)} matches en {elapsed:.3f} s")
        ax.axis('off')
        plt.show()

    return kp1, desc1, kp2, desc2, matches, elapsed

def match_sift_images(path1, path2, sift_params=None, max_ratio=0.8, cross_check=True, plot=True):
    img1 = ensure_gray(io.imread(path1))
    img2 = ensure_gray(io.imread(path2))

    sift_params = sift_params or {}
    t0 = time.time()
    sift = SIFT(**sift_params)

    sift.detect_and_extract(img1)
    kp1, desc1 = sift.keypoints, sift.descriptors

    sift.detect_and_extract(img2)
    kp2, desc2 = sift.keypoints, sift.descriptors

    # Emparellament amb distància euclidiana per a descriptors float
    matches = match_descriptors(desc1, desc2, metric='euclidean', max_ratio=max_ratio, cross_check=cross_check)
    elapsed = time.time() - t0

    if plot:
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_matched_features(img1, img2, keypoints1=kp1, keypoints2=kp2, matches=matches, ax=ax)
        ax.set_title(f"SIFT: {len(matches)} matches en {elapsed:.3f} s")
        ax.axis('off')
        plt.show()

    return kp1, desc1, kp2, desc2, matches, elapsed

# Prova d'execució amb imatges de Starbucks:
try:
    print("Executant ORB...")
    _ = match_orb_images("images_p4/starbucks.jpg", "images_p4/starbucks2.png", plot=False)
    print("Executant SIFT...")
    _ = match_sift_images("images_p4/starbucks.jpg", "images_p4/starbucks2.png", plot=False)
    print("ORB i SIFT funcionen correctament!")
except Exception as e:
    print("Nota:", e)


### ⚖️ Comparativa de Robustesa i Rendiment: SIFT vs ORB

| Propietat | SIFT (Scale-Invariant Feature Transform) | ORB (Oriented FAST & Rotated BRIEF) |
| :--- | :--- | :--- |
| **Tipus de descriptor** | Vector continu float64 de **128 dimensions** | Cadena binària de **256 bits** (booleans) |
| **Mètrica de distància** | Distància Euclidiana ($L_2$) | Distància de Hamming (recompte XOR de bits) |
| **Invariància a la Rotació** | **Molt alta**: orientació dominant calculada per histograma de gradients | **Alta**: orientació per centroid d'intensitat (*oFAST*) |
| **Invariància a l'Escala** | **Molt alta**: espai d'escala continu amb piràmide DoG | **Moderada**: piràmide discreta de resolucions fixes |
| **Robustesa a la Il·luminació** | **Alta**: gradients locals normalitzats i retallats contra reflexos | **Moderada**: comparacions binàries locals de píxels |
| **Velocitat de càlcul** | Més lent (càlcul flotant, optimització necessària) | **Molt ràpid** (adequat per a temps real) |


---
## 16. Anàlisi de Textura i Cerca d'Imatges (CBIR) amb el Banc de Filtres Leung–Malik (LM) [NOU P4]

La textura descriu la disposició espacial de les variacions d'intensitat d'una superfície (rugositat, patrons orientats, freqüència espacial).

### 16.1 Banc de Filtres Leung–Malik (LM) (`util.lm.makeLMfilters`)
El banc de filtres LM conté **48 filtres** de mida 49x49:
- **36 filtres orientats**: primeres i segones derivades de Gaussianes orientades a 6 angles diferents i 3 escales (6x3x2 = 36). Detecten vores i barres orientades.
- **8 filtres Laplacians de Gaussianes (LoG)**: respostes a taques circulars de diferents diàmetres.
- **4 filtres Gaussians**: components de baixa freqüència.

### 16.2 Extracció de Característiques de Textura
Donada una imatge en escala de grisos $I$ i un conjunt de filtres $f_1, f_2, \dots, f_D$:
1. Es convoluciona la imatge amb cada filtre: $r_k = I * f_k$.
2. Es calcula la mitjana del valor absolut de la resposta per capturar l'energia de la textura:
   $$F_k = \frac{1}{HW} \sum_{x, y} |r_k(x, y)|$$
3. El vector característic de la imatge és $f(I) = [F_1, F_2, \dots, F_D]^T \in \mathbb{R}^D$.

### 16.3 Cerca d'Imatges per Contingut (CBIR - Content-Based Image Retrieval)
Mesurem la semblança entre una imatge de consulta (*query*) $x$ i cada imatge $x^m$ de la base de dades mitjançant la distància Euclidiana:
$$d(x, x^m) = \|f(x) - f(x^m)\|_2 = \sqrt{\sum_{d=1}^{D} (f_d(x) - f_d(x^m))^2}$$
Les imatges amb menor distància s'ordenen mitjançant `np.argsort()` per retornar el **Top-$K$** d'imatges més semblants.


In [ ]:
# Import del banc de filtres LM (arxiu util/lm.py)
import sys
if "util" not in sys.path:
    sys.path.append("util")
if "p4_material/util" not in sys.path:
    sys.path.append("p4_material/util")
from util.lm import makeLMfilters

# 1. Generació del banc de filtres
F = makeLMfilters()
print(f"Banc de filtres LM generat: shape {F.shape} (alçada, amplada, nombre_filtres)")

def get_image_features(image, filter_bank, n_filters=8):
    """
    Extreu el vector de característiques de textura convolucionant la imatge amb els n_filters primers filtres.
    """
    from scipy.ndimage import convolve
    features = []
    for k in range(n_filters):
        f_k = filter_bank[:, :, k]
        r_k = convolve(image, f_k)
        # Mitjana de la resposta absoluta
        features.append(np.mean(np.abs(r_k)))
    return np.array(features)

def get_dataset_features(images, filter_bank, n_filters=8):
    """
    Construeix la matriu de característiques per a un conjunt de N imatges.
    Retorna matriu de forma (n_filters, N).
    """
    N = len(images)
    M = np.zeros((N, n_filters))
    for i, img in enumerate(images):
        M[i] = get_image_features(img, filter_bank, n_filters)
    return M.T

# Prova d'extracció de textura sobre la imatge del gos:
if 'dog_gray' in locals():
    dog_240 = resize(dog_gray, (240, 240), anti_aliasing=True)
    feat_vec = get_image_features(dog_240, F, n_filters=8)
    print("Vector de característiques de textura (8 filtres):")
    print(np.round(feat_vec, 5))


---
## 17. `matplotlib.pyplot` & `matplotlib.patches` – Eines de Visualització i Dibuix [AMPLIAT P4]


In [ ]:
# Eines clau de visualització emprades a la P4:
# 1. plt.subplots(nrows, ncols, figsize=(...)): graelles per comparar resultats
# 2. ax.scatter(x, y, s=..., edgecolors='r', facecolors='none'): dibuixar punts clau (CENSURE)
# 3. patches.Rectangle((x, y), width, height, linewidth=..., edgecolor='r', facecolor='none'):
#    dibuixar capses delimitadores per a deteccions (HOG sliding window)
# 4. plot_matched_features(img1, img2, kp1, kp2, matches, ax=...): dibuixar correspondències ORB/SIFT
# 5. ax.imshow(img, cmap='gray', aspect='auto'): visualització de filtres i respostes
# 6. ax.axis('off'): amagar eixos numèrics per visualitzacions netes


---
## 18. `numpy`, `scipy` & eines de sistema (`os`, `glob`) [AMPLIAT P4]


In [ ]:
# Funcions essencials utilitzades a la P4:
# 1. np.linalg.norm(a - b)  → distància euclidiana L2 (per a similitud HOG, descriptors SIFT i cerca CBIR)
# 2. np.argsort(distances)   → índexs que ordenen l'array de menor a major (rànquing Top-K de fotos més semblants)
# 3. np.mean(np.abs(r_k))    → càlcul d'energia de resposta de textura
# 4. np.round(arr, decimals) → arrodoniment per mostrar taules de mètriques
# 5. glob.glob(pattern)      → recupera la llista de rutes de fitxers que compleixen un patró (ex: "images_p4/pets/*.jpg")
# 6. os.path.join(dir, file) → concatenació segura de rutes de directori


---
## 📋 Taula resum de funcions (Pràctica 2 + Pràctica 3 + Pràctica 4)

Aquesta taula recull totes les funcions utilitzades a la col·lecció de pràctiques, indicant biblioteca, mòdul, tipus d'entrada, tipus/forma de retorn i recomanacions clau:

| Funció | Mòdul | Tipus d'entrada | Retorna (dtype · shape · rang) | Notes i Atenció |
|---|---|---|---|---|
| `ensure_gray(img)` | *Funció pròpia* | `uint8` o `float` 2D, RGB o RGBA | `float64` · `(H, W)` · `[0.0, 1.0]` | Suprimeix canal alfa si RGBA i converteix de forma segura |
| `normalize(img)` | *Funció pròpia* | qualsevol array numèric | `float64` · mateixa shape · `[0.0, 1.0]` | Normalització Min-Max segura evitant divisió per zero |
| `io.imread(path, as_gray=...)` | `skimage.io` | Ruta a fitxer (str) | `uint8` · `(H,W,3)` o `(H,W)` · `[0, 255]` | Amb `as_gray=True` retorna `float64` `[0.0, 1.0]` |
| `rgb2gray(img)` | `skimage.color` | `(H,W,3)` qualsevol dtype | `float64` · `(H,W)` · `[0.0, 1.0]` | ⚠️ Falla si té 4 canals (RGBA); usar `ensure_gray` |
| `img_as_float(img)` | `skimage` | qualsevol dtype | `float64` · mateixa shape · `[0.0, 1.0]` | Divideix uint8 entre 255 de forma segura |
| `img_as_ubyte(img)` | `skimage` | `float64` · `[0.0, 1.0]` | `uint8` · mateixa shape · `[0, 255]` | ⚠️ Error o clipping si valors fora de `[0, 1]` |
| `rescale(img, scale, ...)` | `skimage.transform` | qualsevol dtype/shape | `float64` · nova shape · `[0.0, 1.0]` | `preserve_range=True`, `anti_aliasing=True` |
| `resize(img, (H,W), ...)` | `skimage.transform` | qualsevol dtype/shape | `float64` · `(H,W)` o `(H,W,3)` · `[0.0, 1.0]` | Sempre retorna `float64` |
| `rotate(img, angle, ...)` | `skimage.transform` | qualsevol dtype/shape | `float64` · shape pot variar · `[0.0, 1.0]` | `resize=True` evita tallar cantonades en girar |
| `histogram(img, nbins)` | `skimage.exposure` | qualsevol | `(hist, bin_centers)` arrays 1D | Càlcul analític de freqüències per bin |
| `rescale_intensity(img, ...)` | `skimage.exposure` | qualsevol array numèric | `float64` reescalat | Clau per visualitzar imatges HOG (`in_range=(0,10)`) |
| `convolve(img, mask, mode)` | `scipy.ndimage` | qualsevol dtype | mateix dtype que entrada · mateixa shape | Usat per a filtres manuals i banc de filtres LM |
| `convolve2d(img, mask, ...)` | `scipy.signal` | matrius 2D | float64 · forma segons `mode` | Alternativa 2D per a convolució de textures |
| `gaussian(img, sigma)` | `skimage.filters` | qualsevol dtype | `float64` · mateixa shape · `[0.0, 1.0]` | Suavitzat continu gaussià |
| `median(img, footprint)` | `skimage.filters` | qualsevol dtype (millor `uint8`) | **mateix dtype que entrada** · mateixa shape | Ideal per a soroll impulsiu (*salt & pepper*) |
| `disk(r)`, `rectangle`, `diamond` | `skimage.morphology` | Mides enteres | `uint8` de valors binaris `{0, 1}` | Defineix l'element estructurant (*footprint*) |
| `random_noise(img, mode)` | `skimage.util` | qualsevol | `float64` · mateixa shape · `[0.0, 1.0]` | Modes `'gaussian'`, `'s&p'`, etc. |
| `sobel(img)`, `prewitt(img)` | `skimage.filters` | `float64` recomanat | `float64` · mateixa shape | Magnitud total del gradient espacial |
| `canny(img, sigma, ...)` | `skimage.feature` | `uint8` o `float64` grisos | `bool` · mateixa shape · `{True, False}` | Detector de vores multi-etapa amb hysterisi |
| `felzenszwalb(img, ...)` | `skimage.segmentation` | RGB o grisos | `int32` · `(H,W)` · etiquetes de segment | Basat en grafs; sensible al soroll; no cal grisos |
| `slic(img, ...)` | `skimage.segmentation` | RGB qualsevol dtype | `int32` · `(H,W)` · etiquetes de superpíxels | K-Means 5D; robust al soroll; `start_label=1` |
| `quickshift(img, ...)` | `skimage.segmentation` | RGB qualsevol dtype | `int32` · `(H,W)` · etiquetes de segment | Cerca de modes local; `kernel_size`, `max_dist` |
| `mark_boundaries(img, seg)` | `skimage.segmentation` | img qualsevol + seg int | `float64` · `(H,W,3)` · `[0.0, 1.0]` | Retorna sempre imatge RGB amb línies superposades |
| `grid_to_graph(h, w)` | `sklearn.feature_extraction.image` | Dims enteres `h, w` | Matriu dispersa (`coo_matrix`) | Graf de connectivitat 4-veïns de la graella de píxels |
| `AgglomerativeClustering(...)` | `sklearn.cluster` | Matriu de dades `(N, 1)` | Objecte clusterer (`labels_` o `.fit_predict()`) | Clustering jeràrquic restringit pel graf espacial |
| `match_template(img, tpl)` | `skimage.feature` | `float64` grisos | `float64` · `(H-h+1, W-w+1)` · `[-1.0, 1.0]` | Normalized Cross-Correlation (NCC); màxims = coincidència |
| `hog(img, orientations, ...)` | `skimage.feature` | Grisos 2D `[0, 1]` | `(fd, hog_image)` o només `fd` | Descriptor d'Oriented Gradients; per a vianants/objectes |
| `CENSURE(min_scale, ...)` | `skimage.feature` | Classe detector | Instància amb `.keypoints` `(N, 2)` i `.scales` | Detector STAR/DoB; només localitza, no descriu |
| `ORB(n_keypoints, ...)` | `skimage.feature` | Classe detector/extractor | `.keypoints` `(N, 2)` i `.descriptors` `(N, 256)` bool | Descriptor binari molt ràpid basat en oFAST i rBRIEF |
| `SIFT(n_octaves, ...)` | `skimage.feature` | Classe detector/extractor | `.keypoints` `(N, 2)` i `.descriptors` `(N, 128)` float | Invariant a rotació 360° i grans canvis d'escala |
| `match_descriptors(d1, d2, ...)` | `skimage.feature` | Arrays descriptors | `int` · `(M, 2)` · parelles d'índexs | `metric='hamming'` (ORB) o `'euclidean'` (SIFT) |
| `plot_matched_features(...)` | `skimage.feature` | Imatges + keypoints + matches | Dibuixa a l'`ax` de Matplotlib | Línies d'unió entre punts clau emparellats |
| `makeLMfilters()` | `util.lm` | — | `float64` · `(49, 49, 48)` | Banc de 48 filtres de Leung-Malik per a textura |
| `np.median(seq, axis=0)` | `numpy` | Llista o stack de frames 3D/4D | `float64` · mateixes dimensions espacials | Model de fons estàtic temporal |
| `np.abs(test - bck)` | `numpy` | Dos arrays de mateixa shape | Mateixa shape i dtype | Diferència de fotogrames per a moviment |
| `np.linalg.norm(a - b)` | `numpy.linalg` | Vectors o matrius | Escalar float64 | Distància euclidiana L2 (SSD, HOG, SIFT, CBIR) |
| `np.argsort(distances)` | `numpy` | Array 1D de distàncies | Array 1D d'índexs ordenats | Permet obtenir el rànquing Top-K d'imatges similars |
| `np.pad(arr, pad_width, ...)` | `numpy` | Array qualsevol dimensió | Array ampliat amb padding | Alinia mapes de correlació a la mida d'imatge original |
| `np.argwhere(mask)` | `numpy` | Array booleà | Array 2D d'índexs `(N, d)` | Obté les coordenades de pics/deteccions |
| `np.unique(arr)` | `numpy` | Array de valors discrets | Array 1D ordenat de valors únics | Recompte de segments o etiquetes |
| `glob.glob(pattern)` | `glob` | Patró de fitxers (str) | Llista de rutes (list de str) | Cerca fitxers per extensió (ex: `*.jpg`, `*.bmp`) |
| `os.path.join(dir, file)` | `os.path` | Strings de carpetes i fitxers | Ruta normalitzada (str) | Construcció segura de camins de fitxer |
| `patches.Rectangle((x,y),w,h)` | `matplotlib.patches` | Coordenades i mides | Objecte gràfic `Rectangle` | Capsa delimitadora per a deteccions d'objectes |
| `plt.imshow(img, cmap=...)` | `matplotlib.pyplot` | Array 2D o 3D | Objecte `AxesImage` | `cmap='gray'`, `'viridis'`, `'tab10'`, etc. |
| `plt.subplots(r, c, ...)` | `matplotlib.pyplot` | Files i columnes | `(Figure, array of Axes)` | Creació de graelles de visualització |
